In [ ]:
# ==========================================
# STEP 1: INSTALL UN-SLOTH & DEPENDENCIES
# ==========================================
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install trl vllm datasets

import torch
import re
from unsloth import FastLanguageModel
from datasets import Dataset
from trl import GRPOTrainer, GRPOConfig

# ==========================================
# STEP 2: LOAD THE MODEL IN 4-BIT
# ==========================================
# We use a 1.5B model so it safely fits within the free T4 GPU memory limits
model_name = "Qwen/Qwen2.5-1.5B-Instruct"
max_seq_length = 512

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    load_in_4bit = True,            # Reduces memory footprint drastically
    fast_inference = False           # Temporarily set to False to resolve vLLM ImportError due to dependency conflicts
)

# Apply LoRA adapters to allow weight updates on the 4-bit model
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
)

# ==========================================
# STEP 3: CREATE THE REASONING DATASET
# ==========================================
# Simple math problems with concrete, verifiable answers
raw_data = {
    "prompt": [
        "What is 12 multiplied by 5, plus 10?",
        "If 3x + 5 = 20, what is the value of x?",
        "What is the square root of 144?",
        "Solve: 100 divided by 4 minus 5."
    ],
    "answer": ["70", "5", "12", "20"]
}

dataset = Dataset.from_dict(raw_data)

# Inject the system prompt instructing the model to think step-by-step
def format_prompt(example):
    messages = [
        {
            "role": "system",
            "content": "You are a logical reasoning model. Provide your step-by-step thinking inside <think>...</think> tags, and output your final numerical answer at the very end."
        },
        {
            "role": "user",
            "content": example["prompt"]
        }
    ]
    # Apply chat template to convert messages to a single string
    formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return {
        "prompt": formatted_prompt,
        "answer": example["answer"]
    }

dataset = dataset.map(format_prompt)

# ==========================================
# STEP 4: DEFINE VERIFIABLE REWARD FUNCTIONS
# ==========================================

# Reward 1: Verifiable Correctness Reward (The pure RLVR logic)
def accuracy_reward_func(prompts, completions, answer, **kwargs):
    rewards = []
    for completion_text, correct_answer in zip(completions, answer):
        model_output = completion_text.strip()

        # Regex to pull out the very last number from the model's response
        numbers_found = re.findall(r'\d+', model_output)
        final_answer = numbers_found[-1] if numbers_found else None

        # Exact binary check against our static dataset answer
        if final_answer == correct_answer:
            rewards.append(1.0)
        else:
            rewards.append(0.0)
    return rewards

# Reward 2: Format Reward (Teaches the model to use the thinking tags properly)
def format_reward_func(prompts, completions, **kwargs):
    rewards = []
    for completion_text in completions:
        model_output = completion_text.strip()

        # Check if output contains both opening and closing think tags
        if "<think>" in model_output and "</think>" in model_output:
            rewards.append(0.2) # Small bonus reward for formatting correctly
        else:
            rewards.append(0.0)
    return rewards

# ==========================================
# STEP 5: SETUP THE GRPO TRAINING LOOP
# ==========================================
training_args = GRPOConfig(
    output_dir = "./rlvr_colab_outputs",
    learning_rate = 5e-6,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 2,
    num_generations = 4,              # Model will generate 4 unique paths per question
    max_prompt_length = 128,
    max_completion_length = 256,      # Room to think and output the solution
    logging_steps = 1,
    max_steps = 10,                   # Kept short for quick testing in Colab
)

trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [accuracy_reward_func, format_reward_func], # Combining logical and structural rewards
    args = training_args,
    train_dataset = dataset,
)

# Launch RLVR training!
print("Starting RLVR training loop...")
trainer.train()
print("Training complete!")

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-zflcvcvi/unsloth_428b55f0b502400fba5b8631b1e230d6
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-zflcvcvi/unsloth_428b55f0b502400fba5b8631b1e230d6
  Resolved https://github.com/unslothai/unsloth.git to commit 4c06c1dcc771f3d4f2aecc8d3ab77f2e01aa9107
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached transformers-5.5.0-py3-none-any.whl.metadata (32 kB)
Using cached transformers-5.5.0-py3-none-any.whl (10.2 MB)
  Attempting uninstall: transformers
    Found existing installation: transformers 5.10.1
    Uninstalling transformers-5.10.1:
      Successfully uninstalled transformers-5.10.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflict

  Using cached transformers-5.10.1-py3-none-any.whl.metadata (33 kB)
Using cached transformers-5.10.1-py3-none-any.whl (11.0 MB)
  Attempting uninstall: transformers
    Found existing installation: transformers 5.5.0
    Uninstalling transformers-5.5.0:
      Successfully uninstalled transformers-5.5.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth-zoo 2026.6.1 requires transformers!=4.52.0,!=4.52.1,!=4.52.2,!=4.52.3,!=4.53.0,!=4.54.0,!=4.55.0,!=4.55.1,!=4.57.4,!=4.57.5,!=5.0.0,!=5.1.0,<=5.5.0,>=4.51.3, but you have transformers 5.10.1 which is incompatible.


==((====))==  Unsloth 2026.6.1: Fast Qwen2 patching. Transformers: 5.10.1. vLLM: 0.22.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

unsloth/qwen2.5-1.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


...

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4 | Num Epochs = 5 | Total steps = 10
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transfor

Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / accuracy_reward_func / mean,rewards / accuracy_reward_func / std,rewards / format_reward_func / mean,rewards / format_reward_func / std
1,0.000000,1.000000,0.000000,100.000000,71.000000,150.000000,0.000000,100.000000,71.000000,150.000000,0.000007,1.000000,0.000000,0.000000,0.000000
2,0.052379,0.750000,0.288675,162.375000,70.000000,256.000000,0.250000,131.166672,70.000000,249.000000,0.000007,0.750000,0.462910,0.000000,0.000000
3,0.000000,1.000000,0.000000,100.500000,45.000000,179.000000,0.000000,100.500000,45.000000,179.000000,0.000007,1.000000,0.000000,0.000000,0.000000
4,0.031029,0.875000,0.250000,144.125000,70.000000,251.000000,0.000000,144.125000,70.000000,251.000000,0.000008,0.875000,0.353553,0.000000,0.000000
5,0.000000,1.000000,0.000000,122.625000,51.000000,254.000000,0.000000,122.625000,51.000000,254.000000,0.000009,1.000000,0.000000,0.000000,0.000000
6,0.000000,1.000000,0.000000,112.375000,57.000000,148.000000,0.000000,112.375000,57.000000,148.000000,0.000009,1.000000,0.000000,0.000000,0.000000
7,0.000000,1.000000,0.000000,100.625000,62.000000,132.000000,0.000000,100.625000,62.000000,132.000000,0.000012,1.000000,0.000000,0.000000,0.000000
8,0.000000,1.000000,0.000000,92.500000,47.000000,162.000000,0.000000,92.500000,47.000000,162.000000,0.000013,1.000000,0.000000,0.000000,0.000000
9,0.000000,1.000000,0.000000,127.125000,33.000000,195.000000,0.000000,127.125000,33.000000,195.000000,0.000012,1.000000,0.000000,0.000000,0.000000
10,0.000000,1.000000,0.000000,74.375000,49.000000,106.000000,0.000000,74.375000,49.000000,106.000000,0.000008,1.000000,0.000000,0.000000,0.000000


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12

Training complete!
